# 01 — Fetch Training Data from ClinVar

Pulls HBB and HBA1 missense variants from ClinVar via the NCBI E-utilities REST API, maps clinical significance to our 3-class severity label, and writes `data/training_variants.csv`.

**Label mapping** (documented simplification):
| ClinVar significance | → label |
|---|---|
| Benign / Likely benign | `benign` |
| Uncertain significance | `mild` |
| Likely pathogenic / Pathogenic | `severe` |

VUS → mild is conservative. In a production setting, VUS variants would be held out or treated as unlabeled. For this course pipeline they provide additional training signal for the middle class.

In [1]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
import re
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
# Three-letter to one-letter amino acid conversion
AA3_TO_1 = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}

AA1_TO_3 = {v: k for k, v in AA3_TO_1.items()}

UNIPROT_MAP = {'HBB': 'P68871', 'HBA1': 'P69905', 'HBA2': 'P69905'}

SIGNIFICANCE_MAP = {
    'benign': 'benign',
    'likely benign': 'benign',
    'uncertain significance': 'mild',
    'likely pathogenic': 'severe',
    'pathogenic': 'severe',
    'pathogenic/likely pathogenic': 'severe',
    'benign/likely benign': 'benign',
}

In [3]:
ESEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH_URL  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

def search_clinvar(gene: str, retmax: int = 500) -> list[str]:
    """Return ClinVar UIDs for SNVs in the given gene."""
    query = f'"{gene}"[GENE] AND "homo sapiens"[ORGN] AND "single nucleotide variant"[Type]'
    params = {'db': 'clinvar', 'term': query, 'retmax': retmax, 'retmode': 'json'}
    resp = requests.get(ESEARCH_URL, params=params, timeout=30)
    resp.raise_for_status()
    ids = resp.json()['esearchresult']['idlist']
    print(f"  {gene}: {len(ids)} ClinVar UIDs")
    return ids


def fetch_clinvar_xml(uid_list: list[str]) -> ET.Element:
    """Fetch ClinVar VCV XML for a batch of UIDs."""
    params = {
        'db': 'clinvar', 'id': ','.join(uid_list),
        'rettype': 'vcv', 'retmode': 'xml', 'from_esearch': 'yes'
    }
    resp = requests.get(EFETCH_URL, params=params, timeout=60)
    resp.raise_for_status()
    return ET.fromstring(resp.text)


def parse_hgvs_protein(hgvs_p: str):
    """
    Parse an HGVS protein string like 'NP_000509.1:p.Glu7Val' or 'p.Glu7Val'.
    Returns (position: int, wt_aa: str, mut_aa: str) in 3-letter codes, or None.
    """
    m = re.search(r'p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', hgvs_p)
    if not m:
        return None
    wt3, pos, mut3 = m.group(1), int(m.group(2)), m.group(3)
    if wt3 not in AA3_TO_1 or mut3 not in AA3_TO_1:
        return None
    return pos, wt3, mut3


def parse_vcv_record(vcv: ET.Element, gene: str) -> dict | None:
    """
    Extract variant info and clinical significance from a ClinVar VCV XML record.
    Tries multiple XPath patterns to handle different ClinVar format versions.
    Returns a dict or None if the record doesn't meet our criteria.
    """
    # --- Clinical significance ---
    # ClinVar VCV format (2023+): Classifications/GermlineClassification/Description
    # Older format fallbacks follow.
    SIG_XPATHS = [
        './/Classifications/GermlineClassification/Description',
        './/GermlineClassification/Description',
        './/ClinicalSignificance/Description',
        './/Interpretation/Description',
    ]
    sig_raw = None
    for xpath in SIG_XPATHS:
        el = vcv.find(xpath)
        if el is not None and el.text:
            sig_raw = el.text.strip().lower()
            break
    if sig_raw is None:
        return None
    label = SIGNIFICANCE_MAP.get(sig_raw)
    if label is None:
        return None  # skip conflicting / drug response / not provided

    # --- Protein HGVS ---
    # Collect all candidate strings that contain 'p.' and try parsing each.
    hgvs_candidates = []

    # Pattern 1: ProteinExpression/Expression child text (most reliable in VCV format)
    for el in vcv.findall('.//ProteinExpression/Expression'):
        if el.text and 'p.' in el.text:
            hgvs_candidates.append(el.text)

    # Pattern 2: ProteinExpression[@change] attribute (e.g. change="p.Glu7Val")
    for el in vcv.findall('.//ProteinExpression'):
        change = el.get('change', '')
        if change.startswith('p.'):
            hgvs_candidates.append(change)

    # Pattern 3: any HGVS child element text containing 'p.'
    for el in vcv.findall('.//HGVS'):
        for child in el:
            if child.text and 'p.' in child.text:
                hgvs_candidates.append(child.text)

    # Pattern 4: VariationName attribute has protein notation in parentheses
    name_attr = vcv.get('VariationName', '')
    m = re.search(r'\(([^)]*p\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}[^)]*)\)', name_attr)
    if m:
        hgvs_candidates.append(m.group(1))

    hgvs_p = next((s for s in hgvs_candidates if 'p.' in s), None)
    if hgvs_p is None:
        return None

    parsed = parse_hgvs_protein(hgvs_p)
    if parsed is None:
        return None
    position, wt_aa, mut_aa = parsed

    # ClinVar numbers from the initiator Met (position 1), but clinical literature for
    # HBB and HBA1 uses the mature protein numbering (Met removed post-translationally),
    # so subtract 1. E.g. ClinVar "Glu7Val" → clinical "Glu6Val" (HbS).
    if gene in ('HBB', 'HBA1', 'HBA2'):
        position -= 1
    if position <= 0:
        return None

    variant_id = f"{gene}-{wt_aa}{position}{mut_aa}"
    return {
        'variant_id': variant_id,
        'gene': gene,
        'position': position,
        'wt_aa': wt_aa,
        'mut_aa': mut_aa,
        'wt_aa_1': AA3_TO_1[wt_aa],
        'mut_aa_1': AA3_TO_1[mut_aa],
        'severity_label': label,
        'clinvar_significance': sig_raw,
        'uniprot_id': UNIPROT_MAP.get(gene, ''),
    }


def debug_vcv_xml(root: ET.Element, n: int = 1):
    """Print a condensed XML walk of the first n VariationArchive records — use when all_records is empty."""
    archives = root.findall('.//VariationArchive')
    print(f"Root tag: {root.tag}  |  VariationArchive count: {len(archives)}")
    for vcv in archives[:n]:
        print(f"\n=== {vcv.get('VariationName', '?')} ===")
        def walk(el, depth=0):
            text = (el.text or '').strip()[:80]
            attrs = {k: v[:40] for k, v in el.attrib.items()}
            print(f"{'  '*depth}<{el.tag} {attrs}> {repr(text)}")
            for child in list(el)[:6]:  # limit children shown
                walk(child, depth + 1)
        walk(vcv)

In [4]:
GENES = ['HBB', 'HBA1']
BATCH_SIZE = 100

all_records = []
last_root = None  # keep a reference for debugging if needed

for gene in GENES:
    print(f"\nFetching {gene}...")
    uids = search_clinvar(gene, retmax=400)
    time.sleep(0.4)  # NCBI rate limit: 3 req/s without API key

    for i in range(0, len(uids), BATCH_SIZE):
        batch = uids[i:i + BATCH_SIZE]
        print(f"  Fetching batch {i//BATCH_SIZE + 1} ({len(batch)} records)...")
        try:
            root = fetch_clinvar_xml(batch)
            last_root = root
            archives = root.findall('.//VariationArchive')
            parsed_in_batch = 0
            for vcv in archives:
                rec = parse_vcv_record(vcv, gene)
                if rec:
                    all_records.append(rec)
                    parsed_in_batch += 1
            print(f"    → {len(archives)} archives, {parsed_in_batch} parsed")
        except Exception as e:
            print(f"  Warning: batch failed — {e}")
        time.sleep(0.4)

print(f"\nTotal records parsed: {len(all_records)}")
if len(all_records) == 0 and last_root is not None:
    print("\nNo records parsed — printing XML structure of first record to diagnose:")
    debug_vcv_xml(last_root, n=1)


Fetching HBB...
  HBB: 400 ClinVar UIDs
  Fetching batch 1 (100 records)...
    → 100 archives, 40 parsed
  Fetching batch 2 (100 records)...
    → 100 archives, 1 parsed
  Fetching batch 3 (100 records)...
    → 100 archives, 14 parsed
  Fetching batch 4 (100 records)...
    → 100 archives, 0 parsed

Fetching HBA1...
  HBA1: 340 ClinVar UIDs
  Fetching batch 1 (100 records)...
    → 100 archives, 45 parsed
  Fetching batch 2 (100 records)...
    → 100 archives, 25 parsed
  Fetching batch 3 (100 records)...
    → 100 archives, 21 parsed
  Fetching batch 4 (40 records)...
    → 40 archives, 8 parsed

Total records parsed: 154


  HBB: 400 ClinVar UIDs


  Fetching batch 1 (100 records)...


    → 100 archives, 40 parsed


  Fetching batch 2 (100 records)...


...

  Fetching batch 3 (100 records)...


    → 100 archives, 21 parsed


  Fetching batch 4 (40 records)...


    → 40 archives, 8 parsed



Total records parsed: 154


In [5]:
if not all_records:
    raise RuntimeError(
        "No records were parsed. Run the debug_vcv_xml() call above to inspect the XML "
        "structure, then adjust the XPaths in parse_vcv_record accordingly."
    )

df = pd.DataFrame(all_records)
df = df.drop_duplicates(subset='variant_id', keep='first').reset_index(drop=True)
df = df[df['wt_aa_1'] != df['mut_aa_1']]  # drop synonymous (sanity check)

print("Class distribution:")
print(df['severity_label'].value_counts())
print(f"\nTotal unique variants: {len(df)}")
df.head(10)

Class distribution:
severity_label
mild      106
severe     27
benign     20
Name: count, dtype: int64

Total unique variants: 153


,variant_id,gene,position,wt_aa,mut_aa,wt_aa_1,mut_aa_1,severity_label,clinvar_significance,uniprot_id
0,HBB-Gly119Ser,HBB,119,Gly,Ser,G,S,mild,uncertain significance,P68871
1,HBB-Leu78Met,HBB,78,Leu,Met,L,M,mild,uncertain significance,P68871
2,HBB-Val34Leu,HBB,34,Val,Leu,V,L,mild,uncertain significance,P68871
3,HBB-Val54Ile,HBB,54,Val,Ile,V,I,mild,uncertain significance,P68871
4,HBB-Asn108Lys,HBB,108,Asn,Lys,N,K,severe,likely pathogenic,P68871
5,HBB-Phe42Ile,HBB,42,Phe,Ile,F,I,severe,likely pathogenic,P68871
6,HBB-Asn108His,HBB,108,Asn,His,N,H,mild,uncertain significance,P68871
7,HBB-His77Leu,HBB,77,His,Leu,H,L,benign,likely benign,P68871
8,HBB-Leu31Met,HBB,31,Leu,Met,L,M,mild,uncertain significance,P68871
9,HBB-Asp79Glu,HBB,79,Asp,Glu,D,E,mild,uncertain significance,P68871


In [6]:
out_path = DATA_DIR / 'training_variants.csv'
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} variants → {out_path}")

Saved 153 variants → data/training_variants.csv


## Data quality notes

- ClinVar covers well-characterized variants; rarer Indian-specific variants may be absent or labelled VUS
- The VUS → mild mapping is a deliberate simplification documented above
- Conflicting interpretations (e.g., one submitter says Benign, another Pathogenic) are dropped — the `SIGNIFICANCE_MAP` only includes clear single-category strings
- This dataset will be merged with sequence and structural features in notebooks 02 and 03